In [20]:
import numpy as np
from pathlib import Path
import sys

import patsy
import statsmodels.formula.api as smf

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "modeling/data/raw"
CSV_PATH = ROOT / "modeling/data/prcesssed/model_df.csv"
SAVE_MODEL_DF = True

from data_processing import build_model_df

import warnings
warnings.filterwarnings("ignore")

In [17]:
model_df = build_model_df(
    data_dir=DATA_DIR,
    save_csv=SAVE_MODEL_DF,
    csv_path=CSV_PATH,
)

Data processing is now centralized in `src/data_processing.py`.
This notebook uses `build_model_df(...)` to create `model_df`.


In [21]:
# Linear regression
#   - attraction FE: C(ENTITY_DESCRIPTION_SHORT)
#   - DOW FE: C(dow)
#   - season FE: C(season)
#   - COVID interactions with utilization
#   - COVID x attraction FE interaction
#   - Post-COVID x attraction FE interaction
#   - Clustered SEs by attraction, aligned to used rows
#   - Time-based train/test split (80/20, no shuffle)
# ----------------------------

# fill some common missing merges so you don't lose lots of rows
if "attendance" in model_df.columns:
    model_df["attendance"] = model_df["attendance"].fillna(model_df["attendance"].median())
if "scheduled_open_min" in model_df.columns:
    model_df["scheduled_open_min"] = model_df["scheduled_open_min"].fillna(model_df["scheduled_open_min"].median())

# Weather terms (only include if column exists AND has at least some non-missing values)
weather_candidates = ["temp", "rain_1h", "wind_speed", "clouds_all", "humidity"]
weather_terms = [
    c for c in weather_candidates
    if c in model_df.columns and model_df[c].notna().any()
]

# Build formula pieces
base_terms = [
    "utilization",
    "availability",
    "np.log1p(attendance)",
    "scheduled_open_min",
    "nb_units_med",
    "covid",
    "post_covid",
    "utilization:covid",
    "utilization:post_covid",
    "C(dow)",
    "C(season)",
    "C(ENTITY_DESCRIPTION_SHORT)",
    "covid:C(ENTITY_DESCRIPTION_SHORT)",
    "post_covid:C(ENTITY_DESCRIPTION_SHORT)",
]

all_terms = base_terms + weather_terms
formula = "wait_time_avg ~ " + " + ".join(all_terms)

# Time-based split (no shuffle)
model_df_sorted = model_df.sort_values("date").reset_index(drop=True)
split_idx = int(len(model_df_sorted) * 0.8)

train_df = model_df_sorted.iloc[:split_idx].copy()
test_df = model_df_sorted.iloc[split_idx:].copy()

# Build design matrices on train to define columns
y_train, X_train = patsy.dmatrices(formula, data=train_df, return_type="dataframe")
train_used_idx = X_train.index

print(f"Rows in model_df: {len(model_df_sorted):,}")
print(f"Rows used in train (after NA handling): {len(train_used_idx):,}")
print(f"Rows in test before NA handling: {len(test_df):,}")
print(f"Weather terms included: {weather_terms}")

# Align clustering groups to the used rows
# (clusters are based on attraction)
groups_train = train_df.loc[train_used_idx, "ENTITY_DESCRIPTION_SHORT"]

fit = smf.ols(formula, data=train_df.loc[train_used_idx]).fit(
    cov_type="cluster",
    cov_kwds={"groups": groups_train}
)

print(fit.summary())


# Out-of-sample evaluation on test set
# Build test matrices and align columns to train
y_test, X_test = patsy.dmatrices(formula, data=test_df, return_type="dataframe")
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

test_pred = np.dot(X_test, fit.params)
test_actual = y_test.iloc[:, 0].to_numpy()

rmse = np.sqrt(np.mean((test_actual - test_pred) ** 2))
mae = np.mean(np.abs(test_actual - test_pred))

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")


Rows in model_df: 29,847
Rows used in train (after NA handling): 23,283
Rows in test before NA handling: 5,970
Weather terms included: ['temp', 'rain_1h', 'wind_speed', 'clouds_all', 'humidity']
                            OLS Regression Results                            
Dep. Variable:          wait_time_avg   R-squared:                       0.812
Model:                            OLS   Adj. R-squared:                  0.811
Method:                 Least Squares   F-statistic:                 2.298e+06
Date:                Fri, 13 Feb 2026   Prob (F-statistic):           1.00e-70
Time:                        12:42:08   Log-Likelihood:                -81483.
No. Observations:               23283   AIC:                         1.632e+05
Df Residuals:                   23189   BIC:                         1.639e+05
Df Model:                          93                                         
Covariance Type:              cluster                                         
               